# Optimizer comparison on classification using Iris dataset and Logistic Regression

In [1]:
import torch
import torch.nn as nn

from src.quadratic_annealing_optimizer import QuadraticAnnealingOptimizer
from src.models import Logistic
from src.utils import data_load_and_prep, build_sampler
from src.training import train
from src.newton_optimizer import NewtonOptimizer

EXPERIMENT_NAME = "optimizer-comparison-logreg-iris"

In [2]:
train_loader, test_loader = data_load_and_prep("iris", test_size=0.3, random_state=42, batch_size='full')

## Adam

In [3]:
loss_fn = nn.CrossEntropyLoss()
adam_model = Logistic(input_dim=4, output_dim=3)
classical_device = "cuda" 
adam_optimizer = torch.optim.Adam(adam_model.parameters(), 
                             lr=0.01,
                             betas=[0.9, 0.999],
                             )

## Adam optimizaer

In [4]:
train(
    model=adam_model,
    train_loader=train_loader,
    test_loader=test_loader,
    loss_fn=loss_fn,
    optimizer=adam_optimizer,
    c_device=classical_device,
    epochs=50, 
    experiment_name = EXPERIMENT_NAME,
    run_name="adam-optimizer"
)

/home/filip/studia/master/second-order-by-annealer/venv/lib/python3.12/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


NotImplementedError: "nll_loss_forward_reduce_cuda_kernel_2d_index" not implemented for 'Float'

## l-BFGS optimizer

In [11]:
loss_fn = nn.CrossEntropyLoss()
lbfgs_model = Logistic(4, 3)
classical_device = "cuda" 
lbfgs_optimizer = torch.optim.LBFGS(lbfgs_model.parameters(), 
                              lr=0.01,
                              max_iter=1,
                              history_size=100,
                              line_search_fn=None,
                              )

In [12]:
train(
    model=lbfgs_model,
    train_loader=train_loader,
    test_loader=test_loader,
    loss_fn=loss_fn,
    optimizer=lbfgs_optimizer,
    c_device=classical_device,
    epochs=50,
    experiment_name = EXPERIMENT_NAME,
    run_name="lbfgs-optimizer"
)

Epoch 000 | train_loss=1.1220 | train_acc=0.352 | test_loss=1.0730 | test_acc=0.467 | 
Epoch 005 | train_loss=1.0469 | train_acc=0.390 | test_loss=1.0151 | test_acc=0.489 | 
Epoch 010 | train_loss=0.9785 | train_acc=0.419 | test_loss=0.9628 | test_acc=0.489 | 
Epoch 015 | train_loss=0.9183 | train_acc=0.429 | test_loss=0.9159 | test_acc=0.511 | 
Epoch 020 | train_loss=0.8653 | train_acc=0.457 | test_loss=0.8739 | test_acc=0.533 | 
Epoch 025 | train_loss=0.8182 | train_acc=0.505 | test_loss=0.8361 | test_acc=0.533 | 
Epoch 030 | train_loss=0.7761 | train_acc=0.543 | test_loss=0.8019 | test_acc=0.533 | 
Epoch 035 | train_loss=0.7380 | train_acc=0.619 | test_loss=0.7706 | test_acc=0.600 | 


2026/03/20 15:30:11 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.
2026/03/20 15:30:11 WARNING mlflow.utils.requirements_utils: Found torch version (2.10.0+cu126) contains a local version label (+cu126). MLflow logged a pip requirement for this package as 'torch==2.10.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


Epoch 040 | train_loss=0.7034 | train_acc=0.733 | test_loss=0.7419 | test_acc=0.644 | 
Epoch 045 | train_loss=0.6716 | train_acc=0.762 | test_loss=0.7153 | test_acc=0.689 | 
Epoch 049 | train_loss=0.6480 | train_acc=0.781 | test_loss=0.6953 | test_acc=0.711 | 


## Quadriatic Annealing optimizer (cpu)

In [3]:
loss_fn = nn.CrossEntropyLoss()
model = Logistic(4, 3)
classical_device = "cpu"

sampler = build_sampler(mode="simulated")

optimizer = QuadraticAnnealingOptimizer(
    sampler=sampler,
    model=model,
    device="cpu",
    subset_size=12,
    step_size=0.05,
    num_reads=100,
)

In [4]:
train(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    loss_fn=loss_fn,
    optimizer=optimizer,
    c_device=classical_device,
    epochs=50,
    experiment_name = EXPERIMENT_NAME,
    run_name="quadratic-annealer-cpu-optimizer"
)

/home/filip/studia/master/second-order-by-annealer/venv/lib/python3.10/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


BQM construction time: 0.0005 seconds
BQM sampling time: 0.0085 seconds
Parameters update time: 0.0012 seconds
Epoch 000 | train_loss=0.7847 | train_acc=0.686 | test_loss=0.8029 | test_acc=0.667 | 
BQM construction time: 0.0004 seconds
BQM sampling time: 0.0100 seconds
Parameters update time: 0.0010 seconds
BQM construction time: 0.0004 seconds
BQM sampling time: 0.0119 seconds
Parameters update time: 0.0010 seconds
BQM construction time: 0.0004 seconds
BQM sampling time: 0.0105 seconds
Parameters update time: 0.0010 seconds
BQM construction time: 0.0004 seconds
BQM sampling time: 0.0091 seconds
Parameters update time: 0.0010 seconds
BQM construction time: 0.0004 seconds
BQM sampling time: 0.0092 seconds
Parameters update time: 0.0011 seconds
Epoch 005 | train_loss=0.4827 | train_acc=0.867 | test_loss=0.5500 | test_acc=0.800 | 
BQM construction time: 0.0004 seconds
BQM sampling time: 0.0095 seconds
Parameters update time: 0.0010 seconds
BQM construction time: 0.0004 seconds
BQM samplin

2026/03/20 16:19:05 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.
2026/03/20 16:19:05 WARNING mlflow.utils.requirements_utils: Found torch version (2.10.0+cu126) contains a local version label (+cu126). MLflow logged a pip requirement for this package as 'torch==2.10.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


BQM construction time: 0.0004 seconds
BQM sampling time: 0.0092 seconds
Parameters update time: 0.0010 seconds
BQM construction time: 0.0004 seconds
BQM sampling time: 0.0096 seconds
Parameters update time: 0.0010 seconds
Epoch 049 | train_loss=0.0824 | train_acc=0.971 | test_loss=0.1418 | test_acc=0.933 | 


## Quadratic Annealing optimizer (cpu + momentum)

In [3]:
loss_fn = nn.CrossEntropyLoss()
model = Logistic(4, 3)
classical_device = "cpu"

sampler = build_sampler(mode="simulated")

optimizer = QuadraticAnnealingOptimizer(
    sampler=sampler,
    model=model,
    device="cpu",
    subset_size=12,
    step_size=0.05,
    num_reads=100,
    beta=0.9,
)

In [4]:
train(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    loss_fn=loss_fn,
    optimizer=optimizer,
    c_device=classical_device,
    epochs=50,
    experiment_name = EXPERIMENT_NAME,
    run_name="quadratic-annealer-cpu-momentum-optimizer"
)

/home/filip/studia/master/second-order-by-annealer/venv/lib/python3.10/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


Epoch 000 | train_loss=1.0219 | train_acc=0.467 | test_loss=1.0346 | test_acc=0.489 | 
Epoch 005 | train_loss=0.2960 | train_acc=0.867 | test_loss=0.3947 | test_acc=0.822 | 
Epoch 010 | train_loss=0.1361 | train_acc=0.981 | test_loss=0.2468 | test_acc=0.867 | 
Epoch 015 | train_loss=0.0693 | train_acc=0.981 | test_loss=0.1028 | test_acc=0.978 | 
Epoch 020 | train_loss=0.0394 | train_acc=0.981 | test_loss=0.0993 | test_acc=0.978 | 
Epoch 025 | train_loss=0.0281 | train_acc=1.000 | test_loss=0.1660 | test_acc=0.889 | 
Epoch 030 | train_loss=0.0212 | train_acc=1.000 | test_loss=0.1423 | test_acc=0.911 | 
Epoch 035 | train_loss=0.0166 | train_acc=1.000 | test_loss=0.2019 | test_acc=0.911 | 
Epoch 040 | train_loss=0.0132 | train_acc=1.000 | test_loss=0.2107 | test_acc=0.911 | 
Epoch 045 | train_loss=0.0105 | train_acc=1.000 | test_loss=0.2393 | test_acc=0.911 | 
Epoch 049 | train_loss=0.0087 | train_acc=1.000 | test_loss=0.2797 | test_acc=0.911 | 


2026/03/20 19:07:43 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.
2026/03/20 19:07:43 WARNING mlflow.utils.requirements_utils: Found torch version (2.10.0+cu126) contains a local version label (+cu126). MLflow logged a pip requirement for this package as 'torch==2.10.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


## Quadriatic Annealing optimizer (cuda simulated)

In [16]:
loss_fn = nn.CrossEntropyLoss()
model = Logistic(4, 3)
classical_device = "cuda"

sampler = build_sampler(mode="gpu_simulated")

optimizer = QuadraticAnnealingOptimizer(
    sampler=sampler,
    model=model,
    device="cuda",
    subset_size=12,
    step_size=0.05,
    num_reads=100,
)

In [17]:
train(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    loss_fn=loss_fn,
    optimizer=optimizer,
    c_device=classical_device,
    epochs=50,
    experiment_name = EXPERIMENT_NAME,
    run_name="quadratic-annealer-cuda-optimizer"
)

Epoch 000 | train_loss=0.9081 | train_acc=0.610 | test_loss=0.8622 | test_acc=0.667 | 
Epoch 005 | train_loss=0.5302 | train_acc=0.752 | test_loss=0.5673 | test_acc=0.733 | 
Epoch 010 | train_loss=0.3639 | train_acc=0.895 | test_loss=0.4462 | test_acc=0.800 | 
Epoch 015 | train_loss=0.2829 | train_acc=0.924 | test_loss=0.3804 | test_acc=0.844 | 
Epoch 020 | train_loss=0.2310 | train_acc=0.952 | test_loss=0.3267 | test_acc=0.844 | 
Epoch 025 | train_loss=0.1902 | train_acc=0.971 | test_loss=0.2778 | test_acc=0.867 | 
Epoch 030 | train_loss=0.1582 | train_acc=0.971 | test_loss=0.2385 | test_acc=0.867 | 
Epoch 035 | train_loss=0.1339 | train_acc=0.971 | test_loss=0.2098 | test_acc=0.911 | 
Epoch 040 | train_loss=0.1153 | train_acc=0.971 | test_loss=0.1897 | test_acc=0.911 | 
Epoch 045 | train_loss=0.1004 | train_acc=0.971 | test_loss=0.1694 | test_acc=0.933 | 


2026/03/20 15:34:07 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.
2026/03/20 15:34:07 WARNING mlflow.utils.requirements_utils: Found torch version (2.10.0+cu126) contains a local version label (+cu126). MLflow logged a pip requirement for this package as 'torch==2.10.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


Epoch 049 | train_loss=0.0909 | train_acc=0.971 | test_loss=0.1569 | test_acc=0.933 | 


## Newton-Raphson optimizer

In [18]:
loss_fn = nn.CrossEntropyLoss()
newton_model = Logistic(4, 3)
classical_device = "cuda" 
newton_optimizer = NewtonOptimizer(newton_model.parameters(), 
                              lr=0.1,
                              max_iter=1,
                              damping=1e-4,
                              )

In [19]:
train(
    model=newton_model,
    train_loader=train_loader,
    test_loader=test_loader,
    loss_fn=loss_fn,
    optimizer=newton_optimizer,
    c_device=classical_device,
    epochs=50,
    experiment_name = EXPERIMENT_NAME,
    run_name="newton-optimizer"
)

Epoch 000 | train_loss=1.1076 | train_acc=0.333 | test_loss=1.1308 | test_acc=0.356 | 
Epoch 005 | train_loss=0.6536 | train_acc=0.848 | test_loss=0.7127 | test_acc=0.800 | 
Epoch 010 | train_loss=0.4633 | train_acc=0.924 | test_loss=0.5422 | test_acc=0.822 | 
Epoch 015 | train_loss=0.3399 | train_acc=0.981 | test_loss=0.4267 | test_acc=0.822 | 
Epoch 020 | train_loss=0.2354 | train_acc=0.981 | test_loss=0.3106 | test_acc=0.889 | 
Epoch 025 | train_loss=0.1602 | train_acc=0.971 | test_loss=0.2196 | test_acc=0.933 | 
Epoch 030 | train_loss=0.1129 | train_acc=0.971 | test_loss=0.1626 | test_acc=0.933 | 
Epoch 035 | train_loss=0.0826 | train_acc=0.981 | test_loss=0.1296 | test_acc=0.978 | 
Epoch 040 | train_loss=0.0623 | train_acc=0.981 | test_loss=0.1144 | test_acc=0.978 | 
Epoch 045 | train_loss=0.0478 | train_acc=0.990 | test_loss=0.1136 | test_acc=0.956 | 


2026/03/20 15:36:24 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.
2026/03/20 15:36:24 WARNING mlflow.utils.requirements_utils: Found torch version (2.10.0+cu126) contains a local version label (+cu126). MLflow logged a pip requirement for this package as 'torch==2.10.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


Epoch 049 | train_loss=0.0391 | train_acc=0.990 | test_loss=0.1213 | test_acc=0.933 | 
